In [ ]:
from pathlib import Path
import sys

project_root = Path().resolve().parent
sys.path.append(str(project_root)) 

In [ ]:
from src.schema import ConfigSchema
from src.utils.config import read_yaml
from src.utils.training import set_seed

raw_config = read_yaml(project_root / "configs" / "config.yaml")
media_root = project_root / "media"
log_root = project_root / "logs"
config = ConfigSchema(**raw_config)
set_seed(config.experiment.seed)

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

# Настройки для векторного экспорта (текст не переводится в кривые)
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

mpl.rcParams['font.family'] = 'Times New Roman'
mpl.rcParams['font.size'] = 9  # Среднее значение (можно 8 или 10)

# Full model test

In [ ]:
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, ConcatDataset, random_split

from src.dataset.dataset import ConflictEmotionalDataset

dataset_0 = ConflictEmotionalDataset(config.clients[0].dataset)
dataset_1 = ConflictEmotionalDataset(config.clients[1].dataset)
dataset_2 = ConflictEmotionalDataset(config.clients[2].dataset)

train_dataset = ConcatDataset([
    dataset_0.train_dataset,
    dataset_1.train_dataset,
    dataset_2.train_dataset,
])

In [ ]:
def collate_fn(batch):
    xs, ys = zip(*batch)
    
    min_feat_dim = min(x.shape[1] for x in xs)
    
    xs = [x[:, :min_feat_dim].clone().detach().float() if isinstance(x, torch.Tensor)
          else torch.tensor(x[:, :min_feat_dim]).float() for x in xs]
    
    xs = pad_sequence(xs, batch_first=True)
    ys = torch.tensor(ys)
    
    return xs, ys

g = torch.Generator()
g.manual_seed(config.experiment.seed)

train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_dataset, val_dataset = random_split(
    train_dataset, 
    [train_size, val_size],
    generator=g
)

train_loader = DataLoader(
    train_dataset,
    batch_size=(
        len(config.clients)
        * config.clients[0].runtime.batch_size
    ),
    shuffle=True,
    collate_fn=collate_fn,
    generator=g,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=(
        len(config.clients)
        * config.clients[0].runtime.batch_size
    ),
    shuffle=False,
    collate_fn=collate_fn,
    generator=g
)

In [ ]:
test_loaders = {
    f"loader_{i}": DataLoader(ds.test_dataset, batch_size=24, collate_fn=collate_fn)
    for i, ds in enumerate([dataset_0, dataset_1, dataset_2])
}

In [ ]:
from src.model.speech_model import SpeechRecognitionModel
import torch.nn as nn
import torch.optim as optim

from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

pos_weights = torch.tensor(config.split_server.model.pos_weight).to(device)
model = SpeechRecognitionModel(
    input_channels=dataset_0.train_dataset.data.shape[1],
    server_side_model_type='cnn_birnn'
).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights).to(device)

optimizer = optim.Adam(
    params=model.parameters(),
    lr=config.split_server.model.learning_rate
)

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import torch
from tqdm import tqdm
import numpy as np

# Training parameters
epochs = 89
best_val_f1 = 0.0
best_model_state = None

train_losses = []
val_losses = []

train_metrics = {
    "f1": [],
    "accuracy": []
}
val_metrics = {
    "f1": [],
    "accuracy": [],
    "recall": [],
    "precision": []
}

# Training loop
for epoch in range(epochs):
    # ========== TRAINING PHASE ==========
    model.train()
    epoch_loss = 0
    all_preds = []
    all_labels = []
    
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
        inputs, labels = batch
        inputs = inputs.to(device)
        labels = labels.to(device).float()
        labels = labels.reshape(-1)
        
        optimizer.zero_grad()
        
        outputs = model(inputs).reshape(-1)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
        with torch.no_grad():
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).int()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Training metrics
    train_loss = epoch_loss / len(train_loader)
    train_losses.append(train_loss)
    train_f1 = f1_score(all_labels, all_preds, average='binary')
    train_acc = accuracy_score(all_labels, all_preds)
    train_metrics["f1"].append(train_f1)
    train_metrics["accuracy"].append(train_acc)
    
    # ========== VALIDATION PHASE ==========
    model.eval()
    val_loss = 0
    val_preds = []
    val_labels = []
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
            inputs, labels = batch
            inputs = inputs.to(device)
            labels = labels.to(device).float()
            labels = labels.reshape(-1)
            
            outputs = model(inputs).reshape(-1)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).int()
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels.cpu().numpy())
    
    # Validation metrics
    val_loss = val_loss / len(val_loader)
    val_losses.append(val_loss)
    val_f1 = f1_score(val_labels, val_preds, average='binary')
    val_acc = accuracy_score(val_labels, val_preds)
    val_precision = precision_score(val_labels, val_preds, average='binary')
    val_recall = recall_score(val_labels, val_preds, average='binary')
    val_metrics["accuracy"].append(val_acc)
    val_metrics["f1"].append(val_f1)
    val_metrics["precision"].append(val_precision)
    val_metrics["recall"].append(val_recall)
    
    # Print metrics
    print(f"\nEpoch {epoch+1}/{epochs}")
    print(f"  Train - Loss: {train_loss:.4f}, F1: {train_f1:.4f}, Acc: {train_acc:.4f}")
    print(f"  Val   - Loss: {val_loss:.4f}, F1: {val_f1:.4f}, Acc: {val_acc:.4f}")
    print(f"          Precision: {val_precision:.4f}, Recall: {val_recall:.4f}")
    print("-" * 60)

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, average_precision_score, accuracy_score
import numpy as np

model.eval()

threshold= 0.5
total_loss = 0
all_probs = []
all_labels = []
dataset_names = []  # Optional: track which dataset each prediction comes from

with torch.no_grad():
    for test_loader in test_loaders:
        for inputs, labels in tqdm(test_loaders[test_loader], desc=f"Testing {test_loader}"):
            inputs = inputs.to(device)
            labels = labels.to(device).float()

            outputs = model(inputs).reshape(-1)
            labels = labels.reshape(-1)

            loss = criterion(outputs, labels)
            total_loss += loss.item()

            probs = torch.sigmoid(outputs)

            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            dataset_names.extend([test_loader] * len(probs))  # Optional

# --- numpy ---
all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

# Calculate metrics
average_loss = total_loss / sum(len(test_loaders[test_loader]) for test_loader in test_loaders)
print(f"Test Loss: {average_loss:.4f}")

# For binary classification
predictions = (all_probs > threshold).astype(int)

precision = precision_score(all_labels, predictions)
recall = recall_score(all_labels, predictions)
f1 = f1_score(all_labels, predictions)
acc = accuracy_score(all_labels, predictions)

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Accuracy: {acc:.4f}")

# Optional: Calculate per-dataset metrics
if dataset_names:
    unique_datasets = set(dataset_names)
    print("\nPer-dataset metrics:")
    for dataset in unique_datasets:
        mask = np.array(dataset_names) == dataset
        dataset_labels = all_labels[mask]
        dataset_probs = all_probs[mask]
        dataset_preds = (dataset_probs > threshold).astype(int)
        
        print(f"\n{dataset}:")
        print(f"  Precision: {precision_score(dataset_labels, dataset_preds):.4f}")
        print(f"  Recall: {recall_score(dataset_labels, dataset_preds):.4f}")
        print(f"  F1: {f1_score(dataset_labels, dataset_preds):.4f}")
        print(f"  Acc: {accuracy_score(dataset_labels, dataset_preds):.4f}")

In [ ]:
plt.figure(figsize=(10, 6))

plt.plot(train_losses, label='Обучающая выборка', linewidth=2, alpha=0.9, marker='o')
plt.plot(val_losses, label='Валидационная выборка', linewidth=2, color='orangered', linestyle='--', alpha=0.9, marker='o')

plt.title('Динамика функции потерь в процессе обучения')
plt.xlabel('Раунд')
plt.ylabel('Ср. ошибка')

plt.grid(True, alpha=0.6, linestyle='--', linewidth=0.5)

plt.legend(loc='upper right', fontsize=11, frameon=True, fancybox=True, shadow=True)

plt.tight_layout()
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)

min_train_idx = train_losses.index(min(train_losses)) if train_losses else 0
min_val_idx = val_losses.index(min(val_losses)) if val_losses else 0
plt.scatter(min_train_idx, min(train_losses), color='royalblue', s=20, zorder=5, marker="*")
plt.scatter(min_val_idx, min(val_losses), color='orangered', s=20, zorder=5, marker="*")
plt.ylim(0, 1.5)

#Экспорт в векторные форматы
plt.savefig(f'{media_root}/full_train_val_plots.pdf', format='pdf', bbox_inches='tight', dpi=300)
plt.savefig(f'{media_root}/full_train_val_plots.svg', format='svg', bbox_inches='tight')
plt.savefig(f'{media_root}/full_train_val_plots.eps', format='eps', bbox_inches='tight')

plt.show()

In [ ]:
threshold = 0.5

preds = (all_probs > threshold).astype(int)

# --- метрики ---
f1 = f1_score(all_labels, preds)
precision = precision_score(all_labels, preds)
recall = recall_score(all_labels, preds)
pr_auc = average_precision_score(all_labels, all_probs)

avg_loss = total_loss / len(test_loader)

print(f"Test Loss: {avg_loss:.4f}")
print(f"F1: {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"PR-AUC: {pr_auc:.4f}")

In [ ]:
import matplotlib.pyplot as plt
plt.hist(all_probs, bins=20)